# 05 | Coarse estimates and fine relative wetness indicator
We map trained-model predictions only on the SMAP grid. A separate 250 m relative contrast uses the fitted fusion ridge's linear score applied to fine features and subtracts its coarse-area mean, then divides by scene-wide contrast standard deviation. Positive values indicate a higher fitted score than the corresponding coarse cell; negative values indicate lower. It is **dimensionless, exploratory and not a validated 250 m soil-moisture retrieval**. The score is not proof of irrigation or crop stress.

Select a dry and wet month from January–March and June–September 2025, respectively, using only predictor support (not SMAP values or model performance).

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

In [ ]:
conf=json.loads((ROOT/'data'/'configuration.json').read_text())
fine,coarse=conf['fine'],conf['coarse']
land=sw.read_tif(ROOT/'data'/'land.tif',fine,['land_fraction','crop_fraction'])

In [ ]:
bundle=joblib.load(ROOT/'models'/'retrieval.joblib') if (ROOT/'models'/'retrieval.joblib').exists() else None
assert bundle is not None,'Run notebook 04 first.'
assert bundle['sklearn']==version('scikit-learn'),'Use the same scikit-learn version as notebook 04.'
df=pd.read_csv(ROOT/'data'/'coarse_monthly_table.csv')
opts=df[df.core].groupby('month').predictable.sum().rename('supported').reset_index()
opts['year']=pd.to_datetime(opts.month).dt.year; opts['m']=pd.to_datetime(opts.month).dt.month
dates=[]
for season,months in [('dry',[1,2,3]),('wet',[6,7,8,9])]:
    best=opts[(opts.year==2025)&opts.m.isin(months)].sort_values(['supported','month'],ascending=[False,True]).iloc[0]
    if best.supported>0: dates.append((season,best.month))
print('Selected from coverage:',dates)
assert dates,'No supported 2025 date; report missingness.'

In [ ]:
selected=bundle['selected']; cols,model=bundle['models'][selected]
ridge=bundle['models']['fusion_ridge'][1]; summary=[]
for season,key in dates:
    sar=sw.read_tif(ROOT/'data'/f'sar_{key}.tif',fine,sw.S1)
    optical=sw.read_tif(ROOT/'data'/f'optical_{key}.tif',fine,sw.S2)
    smap=sw.read_tif(ROOT/'data'/f'smap_{key}.tif',coarse,['sm','smap_days'])
    t,valid=sw.summarize_month(sar,optical,land,smap,fine,coarse,key); t=sw.add_season(t)
    supported=t.predictable.copy()
    for c,(lo,hi) in bundle['training_bounds'].items(): supported &= t[c].between(lo,hi)
    # The bounds screen is conservative and marginal, not multivariate OOD detection.
    p=np.full(len(t),np.nan,dtype='float32'); p[supported]=model.predict(t.loc[supported,cols]) if supported.any() else np.array([])
    p[(p<0)|(p>.65)]=np.nan; estimate=p.reshape(sw.shape(coarse))
    paired=np.isfinite(estimate)&t.eligible.to_numpy().reshape(sw.shape(coarse)); ref=np.where(paired,smap[0],np.nan); diff=np.where(paired,estimate-ref,np.nan)
    contrast=sw.fine_contrast(ridge,sar,optical,valid,fine,coarse,supported.to_numpy().reshape(sw.shape(coarse)))
    crop_contrast=np.where(land[1]>=.5,contrast,np.nan)
    for name,a,g in [('estimated_sm',estimate,coarse),('smap_paired',ref,coarse),('difference',diff,coarse),('coverage',t.coverage.to_numpy().reshape(sw.shape(coarse)),coarse),('relative_contrast',crop_contrast,fine),('vv_db',sw.db(sar[0]),fine),('ndvi',optical[0],fine)]:
        sw.write_tif(ROOT/'qgis'/f'{name}_{key}.tif',a,g,[name])
    summary.append(dict(season=season,month=key,model=selected,predictable=int(t.predictable.sum()),within_bounds=int(supported.sum()),mapped=int(np.isfinite(estimate).sum()),paired=int(paired.sum()),crop_contrast_pixels=int(np.isfinite(crop_contrast).sum())))
    fig,axs=plt.subplots(1,4,figsize=(15,4),constrained_layout=True)
    for ax,a,title,cm,lim in zip(axs,[estimate,ref,diff,crop_contrast],['Coarse estimate','SMAP paired','Estimate − SMAP','250 m relative score (crop ≥50%)'],['YlGnBu','YlGnBu','RdBu','RdBu'],[(0,.45),(0,.45),(-.15,.15),(-2,2)]):
        h=ax.imshow(a,cmap=cm,vmin=lim[0],vmax=lim[1]); ax.set_title(title); ax.axis('off'); fig.colorbar(h,ax=ax,shrink=.6)
    fig.suptitle(key); fig.savefig(ROOT/'figures'/f'05_maps_{key}.png',dpi=150); plt.show()
summary=pd.DataFrame(summary); display(summary); summary.to_csv(ROOT/'outputs'/'mapping_summary.csv',index=False)

**Checkpoint:** coarse maps may be absent if features are outside their training bounds. Fine contrast is a model-based relative indicator, plotted only where crop fraction is ≥50%; its values are standardized per scene, so magnitudes across months are not directly comparable. Never label it as measured soil moisture or use it for irrigation prescriptions.